In [42]:
import sqlite3
import hashlib
import os
import subprocess
import secrets      
import re          
import logging      
import json         


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

print('Imports loaded')
print('Logger configured')

Imports loaded
Logger configured


In [43]:

DB_USER     = os.environ.get('DB_USER', 'NOT_SET')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'NOT_SET')
SECRET_KEY  = os.environ.get('SECRET_KEY', secrets.token_hex(32)) 

print(f'DB_USER    : {DB_USER}')
print(f'DB_PASSWORD: {"*" * len(DB_PASSWORD)}')
print(f'SECRET_KEY : {SECRET_KEY[:8]}... (truncated for safety)')
print()
print('Credentials loaded from environment — not hardcoded!')
print('Set real values with: os.environ["DB_USER"] = "youruser"')

DB_USER    : NOT_SET
DB_PASSWORD: *******
SECRET_KEY : 50d352ab... (truncated for safety)

Credentials loaded from environment — not hardcoded!
Set real values with: os.environ["DB_USER"] = "youruser"


In [44]:
CONN = sqlite3.connect(':memory:')

def setup_db():
    CONN.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id       INTEGER PRIMARY KEY,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            email    TEXT NOT NULL,
            role     TEXT NOT NULL DEFAULT 'user'
        )
    """)
    CONN.commit()
    print('Secure database created')

setup_db()

Secure database created


In [45]:

def login_secure(username, password):
    hashed = hash_password_secure(password)
   
    result = CONN.execute(
        'SELECT * FROM users WHERE username = ? AND password = ?',
        (username, hashed)
    ).fetchone()
    return result

print('Login uses parameterized queries')
print()
print('SQL injection attempt: \' OR \'1\'=\'1\' --')
result = login_secure("' OR '1'='1' --", 'anything')
print(f'Result: {result}')
print()
print('Returns None — injection attempt completely neutralized!')

Login uses parameterized queries

SQL injection attempt: ' OR '1'='1' --
Result: None

Returns None — injection attempt completely neutralized!


In [46]:

def hash_password_secure(password: str) -> str:
    salt = os.environ.get('PASSWORD_SALT', 'codealpha_secure_salt_2026')
    return hashlib.pbkdf2_hmac(
        'sha256',
        password.encode('utf-8'),
        salt.encode('utf-8'),
        iterations=260000      
    ).hex()

import hashlib as _hl
def hash_md5_demo(p): return _hl.md5(p.encode()).hexdigest()

pwd = 'password123'
print(f'Password       : {pwd}')
print(f'MD5 (insecure) : {hash_md5_demo(pwd)}')
print(f'PBKDF2 (secure): {hash_password_secure(pwd)}')
print()
print('PBKDF2 is slow by design — makes brute force attacks impractical')
print('Salt ensures identical passwords produce different hashes')

Password       : password123
MD5 (insecure) : 482c811da5d5b4bc6d497ffa98491e38
PBKDF2 (secure): 79ec9bf5a5bc581aa7ab9569020702d8e0ac5d1140c7fbfda5dfdf8541ab09f6

PBKDF2 is slow by design — makes brute force attacks impractical
Salt ensures identical passwords produce different hashes


In [47]:

def validate_username(username: str) -> bool:
    # Only alphanumeric + underscore, 3-30 chars
    return bool(re.match(r'^[a-zA-Z0-9_]{3,30}$', username))

def validate_email(email: str) -> bool:
    return bool(re.match(r'^[\w\.\-]+@[\w\.\-]+\.\w{2,}$', email))

def validate_password(password: str) -> bool:
    if len(password) < 8:              return False
    if not any(c.isupper() for c in password): return False
    if not any(c.isdigit() for c in password): return False
    return True


tests = [
    ('Username', validate_username, ['alice_01', 'a', 'invalid user!', 'valid_user_123']),
    ('Email',    validate_email,    ['alice@gmail.com', 'not-an-email', 'a@b.co']),
    ('Password', validate_password, ['weak', 'Password1', 'allowercase1', 'NoDigits!']),
]

for label, fn, cases in tests:
    print(f'--- {label} Validation ---')
    for val in cases:
        icon = 'Good' if fn(val) else 'Bad'
        print(f'  {icon} "{val}"')
    print()

--- Username Validation ---
  Good "alice_01"
  Bad "a"
  Bad "invalid user!"
  Good "valid_user_123"

--- Email Validation ---
  Good "alice@gmail.com"
  Bad "not-an-email"
  Good "a@b.co"

--- Password Validation ---
  Bad "weak"
  Good "Password1"
  Bad "allowercase1"
  Bad "NoDigits!"



## ✅ Cell 7 — FIX 4+5: Secure Registration (Validation + Safe Logging)

In [48]:

def register_user_secure(username: str, password: str, email: str, role: str = 'user'):
    if not validate_username(username):
        raise ValueError('Invalid username. Use 3-30 alphanumeric chars or underscores.')
    if not validate_email(email):
        raise ValueError('Invalid email address.')
    if not validate_password(password):
        raise ValueError('Password must be 8+ chars with at least one uppercase and digit.')
    if role not in ('user', 'moderator'):
        raise ValueError('Invalid role — cannot self-assign admin.')

    hashed = hash_password_secure(password)
    CONN.execute(
        'INSERT INTO users (username, password, email, role) VALUES (?, ?, ?, ?)',
        (username, hashed, email, role)
    )
    CONN.commit()
   
    logger.info(f'New user registered: {username} ({email})')

# Valid registration
try:
    register_user_secure('alice', 'Password1', 'alice@example.com')
    print(' Valid user registered successfully')
except ValueError as e:
    print(f'❌ {e}')

print()

# Invalid attempts
invalid_tests = [
    ('a',       'Password1',  'a@b.com',        'user',  'Username too short'),
    ('bob',     'weak',       'bob@b.com',       'user',  'Weak password'),
    ('charlie', 'Password1',  'not-an-email',    'user',  'Invalid email'),
    ('dave',    'Password1',  'dave@example.com','admin', 'Cannot self-assign admin'),
]

for uname, pwd, email, role, desc in invalid_tests:
    try:
        register_user_secure(uname, pwd, email, role)
        print(f'  Registered (unexpected): {uname}')
    except ValueError as e:
        print(f' Blocked ({desc}): {e}')

2026-05-04 21:29:55,454 [INFO] New user registered: alice (alice@example.com)


 Valid user registered successfully

 Blocked (Username too short): Invalid username. Use 3-30 alphanumeric chars or underscores.
 Blocked (Weak password): Password must be 8+ chars with at least one uppercase and digit.
 Blocked (Invalid email): Invalid email address.
 Blocked (Cannot self-assign admin): Invalid role — cannot self-assign admin.


In [49]:

ALLOWED_HOST = re.compile(r'^[a-zA-Z0-9\.\-]{1,253}$')

def ping_host_secure(host: str):
    if not ALLOWED_HOST.match(host):
        raise ValueError(f'Invalid hostname: {host!r}')
   
    print(f'  Command list: ["ping", "-c", "1", "{host}"]  (shell=False)')
    


try:
    ping_host_secure('8.8.8.8')
    print('  Valid hostname accepted')
except ValueError as e:
    print(f'  Error {e}')

print()

# Injection attempts
attacks = ['8.8.8.8; cat /etc/passwd', '8.8.8.8 && rm -rf /', '$(whoami)', '`id`']
for attack in attacks:
    try:
        ping_host_secure(attack)
    except ValueError as e:
        print(f' Blocked injection: {attack!r}')

  Command list: ["ping", "-c", "1", "8.8.8.8"]  (shell=False)
  Valid hostname accepted

 Blocked injection: '8.8.8.8; cat /etc/passwd'
 Blocked injection: '8.8.8.8 && rm -rf /'
 Blocked injection: '$(whoami)'
 Blocked injection: '`id`'


In [50]:

def read_user_file_secure(filename: str):
    base_dir = os.path.abspath('/var/app/uploads/')
    
    safe_path = os.path.abspath(os.path.join(base_dir, os.path.basename(filename)))

    if not safe_path.startswith(base_dir):
        raise PermissionError(f'Path traversal detected! Attempted: {safe_path}')

    print(f'  Safe path resolved: {safe_path}')
   

# Normal file
try:
    read_user_file_secure('profile.jpg')
    print('Normal filename accepted')
except PermissionError as e:
    print(f'Error {e}')

print()

# Traversal attacks
attacks = ['../../etc/passwd', '../secret.txt', '/etc/shadow', '....//....//etc/passwd']
for attack in attacks:
    try:
        read_user_file_secure(attack)
    except PermissionError as e:
        print(f' Blocked traversal: {attack!r}')

  Safe path resolved: C:\var\app\uploads\profile.jpg
Normal filename accepted

  Safe path resolved: C:\var\app\uploads\passwd
  Safe path resolved: C:\var\app\uploads\secret.txt
  Safe path resolved: C:\var\app\uploads\shadow
  Safe path resolved: C:\var\app\uploads\passwd


In [51]:

def generate_token_secure() -> str:
    return secrets.token_hex(32)   

import random as _rng
print('Insecure (random module) — only 900,000 possibilities:')
for i in range(3):
    print(f'  ⚠️  {_rng.randint(100000, 999999)}')

print()
print('Secure (secrets module) — 2^256 possibilities:')
for i in range(3):
    print(f' {generate_token_secure()}')

print()
print(f'Token length: {len(generate_token_secure())} hex chars = 256 bits of entropy')

Insecure (random module) — only 900,000 possibilities:
  ⚠️  473308
  ⚠️  948983
  ⚠️  600796

Secure (secrets module) — 2^256 possibilities:
 b907f788bb08f41ad28e942fb85ff7f30a7dd95e3412931c2240f9bf15a92d99
 e44245c025a950fbd39448e669c5d56914831a7cd9801c99f8852376a5e96db8
 18c26e50cc08601114606301c31056bf8e6c9b9cc149223c6f9495c40b89fa28

Token length: 64 hex chars = 256 bits of entropy


In [52]:

def get_user_secure(user_id: int):
    if not isinstance(user_id, int):
        raise TypeError('user_id must be an integer.')
    try:
        result = CONN.execute(
            'SELECT id, username, email, role FROM users WHERE id = ?',
            (user_id,)
        ).fetchone()
        return result
    except Exception:
        logger.error('Database error in get_user', exc_info=True)  
        return None   


result = get_user_secure(1)
print(f'User lookup result: {result}')

print()


try:
    get_user_secure('not_an_int')
except TypeError as e:
    print(f'Type error caught cleanly: {e}')

print()
print('Errors are logged internally — users never see database internals')

User lookup result: (1, 'alice', 'alice@example.com', 'user')

Type error caught cleanly: user_id must be an integer.

Errors are logged internally — users never see database internals


In [53]:

def load_session_secure(session_data: str) -> dict:
    try:
        data = json.loads(session_data)
        if not isinstance(data, dict):
            raise ValueError('Invalid session format.')
        return data
    except (json.JSONDecodeError, ValueError) as e:
        logger.warning(f'Invalid session data: {e}')
        return {}  


valid_session = json.dumps({'user_id': 1, 'role': 'user'})
result = load_session_secure(valid_session)
print(f' Valid session loaded: {result}')

print()


bad_inputs = ['{invalid json}', '"just a string"', '12345']
for bad in bad_inputs:
    result = load_session_secure(bad)
    print(f'Bad input handled safely: {bad!r} → {result}')

print()
print('JSON cannot execute code — pickle can. Always prefer JSON for data exchange.')

2026-05-04 21:29:56,651 [WARNING] Invalid session data: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
2026-05-04 21:29:56,652 [WARNING] Invalid session data: Invalid session format.
2026-05-04 21:29:56,652 [WARNING] Invalid session data: Invalid session format.


 Valid session loaded: {'user_id': 1, 'role': 'user'}

Bad input handled safely: '{invalid json}' → {}
Bad input handled safely: '"just a string"' → {}
Bad input handled safely: '12345' → {}

JSON cannot execute code — pickle can. Always prefer JSON for data exchange.


In [54]:
fixes = [
    ('VUL-01', 'SQL Injection',             'CRITICAL', 'Parameterized queries with ? placeholders'),
    ('VUL-02', 'Hardcoded Credentials',      'CRITICAL', 'os.environ.get() — load from environment'),
    ('VUL-03', 'Command Injection',          'CRITICAL', 'Regex whitelist + shell=False + list args'),
    ('VUL-04', 'Insecure Deserialization',   'CRITICAL', 'Replaced pickle with json.loads()'),
    ('VUL-05', 'Weak Hashing (MD5)',          'HIGH',    'pbkdf2_hmac SHA-256 with 260k iterations'),
    ('VUL-06', 'Path Traversal',             'HIGH',     'os.path.abspath() + startswith() jail'),
    ('VUL-07', 'Insecure Random Token',      'HIGH',     'secrets.token_hex(32) — 256-bit entropy'),
    ('VUL-08', 'Sensitive Data in Logs',     'HIGH',     'logging module — never log passwords/hashes'),
    ('VUL-09', 'Missing Input Validation',   'MEDIUM',   'Regex validators for all user inputs'),
    ('VUL-10', 'Verbose Error Messages',     'MEDIUM',   'Return None to caller, log details internally'),
]

sev_icon = {'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡'}

print('=' * 72)
print('  CodeAlpha Task 3 — Secure Code Review: All Fixes Applied')
print('=' * 72)
print(f'{"ID":<10}{"Severity":<12}{"Fix Applied"}')
print('-' * 72)
for vid, vuln, sev, fix in fixes:
    print(f'{vid:<10}{sev_icon[sev]} {sev:<10}{fix}')
print('=' * 72)
print(f'\n   All {len(fixes)} vulnerabilities remediated in secure_app.ipynb')
print(' GitHub: CodeAlpha_SecureCodingReview')

  CodeAlpha Task 3 — Secure Code Review: All Fixes Applied
ID        Severity    Fix Applied
------------------------------------------------------------------------
VUL-01    🔴 CRITICAL  Parameterized queries with ? placeholders
VUL-02    🔴 CRITICAL  os.environ.get() — load from environment
VUL-03    🔴 CRITICAL  Regex whitelist + shell=False + list args
VUL-04    🔴 CRITICAL  Replaced pickle with json.loads()
VUL-05    🟠 HIGH      pbkdf2_hmac SHA-256 with 260k iterations
VUL-06    🟠 HIGH      os.path.abspath() + startswith() jail
VUL-07    🟠 HIGH      secrets.token_hex(32) — 256-bit entropy
VUL-08    🟠 HIGH      logging module — never log passwords/hashes
VUL-09    🟡 MEDIUM    Regex validators for all user inputs
VUL-10    🟡 MEDIUM    Return None to caller, log details internally

   All 10 vulnerabilities remediated in secure_app.ipynb
 GitHub: CodeAlpha_SecureCodingReview
